In [1]:
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"]="0"

import numpy as np
import tensorflow as tf
from tabulate import tabulate
from tensorflow.python.client import device_lib
from tqdm import tqdm
import time

from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)


2025-10-22 03:40:42.033274: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761084642.044968  124927 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761084642.048676  124927 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761084642.058323  124927 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761084642.058340  124927 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1761084642.058341  124927 computation_placer.cc:177] computation placer alr

In [2]:

def print_gpu_details():
    """Prints all available GPUs in a formatted table with key details"""
    # Get list of all devices
    devices = device_lib.list_local_devices()
    gpu_details = []
    
    for device in devices:
        if device.device_type == 'GPU':
            # Extract details from the device description string
            desc = device.physical_device_desc
            details = {
                'Device ID': device.name.split(':')[-1],
                'Name': desc.split('name: ')[1].split(',')[0] if 'name: ' in desc else 'Unknown',
                'Memory (GB)': f"{device.memory_limit / (1024**3):.2f}",
                'PCI Bus ID': desc.split('pci bus id: ')[1].split(',')[0] if 'pci bus id: ' in desc else 'Unknown',
                'GFX Version': os.environ.get('HSA_OVERRIDE_GFX_VERSION', 'Native')
            }
            gpu_details.append(details)
    
    # Print table if GPUs found
    if gpu_details:
        print("\n" + "="*85)
        print("ACTIVE GPU CONFIGURATION".center(85))
        print("="*85)
        print(tabulate(gpu_details, headers="keys", tablefmt="grid"))
        print("="*85+ "\n")
    else:
        print("No GPU devices found!")


def configure_gpu(vram_limit):
    """Configure GPU settings with optional GFX override"""
    
    # Verify GPU availability
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        try:
            # Limit VRAM on the specified GPU
            
            for gid in range(len(gpus)):
                
                tf.config.experimental.set_virtual_device_configuration(
                    gpus[gid],
                    [tf.config.experimental.VirtualDeviceConfiguration(
                        memory_limit=vram_limit[gid] * 1024)]  # Convert GB to MB
                )
                print(f"GPU {gid} VRAM limited to {vram_limit[gid]}GB")
                
        except RuntimeError as e:
            print(f"Error setting VRAM limit: {e}")
    
    if not gpus:
        raise RuntimeError(f"No GPU found")
    
    for gid in range(len(gpus)):
        print(f"Configured GPU {gid}: {tf.config.experimental.get_device_details(gpus[gid])}") 
        
        with tf.device('/GPU:'+str(gid)):  # Force GPU usage
            x = tf.ones((1, 1))    # Smallest possible tensor
            y = x + 1              # Simple operation
            y.numpy()              # Force execution

In [3]:
VRAM = [3.9]

configure_gpu(VRAM)

print_gpu_details()

GPU 0 VRAM limited to 3.9GB
Configured GPU 0: {'compute_capability': (8, 6), 'device_name': 'NVIDIA GeForce RTX 3050 Laptop GPU'}

                               ACTIVE GPU CONFIGURATION                              
+-------------+------------------------------------+---------------+--------------+---------------+
|   Device ID | Name                               |   Memory (GB) | PCI Bus ID   | GFX Version   |
+=============+====================================+===============+==============+===============+
|           0 | NVIDIA GeForce RTX 3050 Laptop GPU |           3.9 | 0000:01:00.0 | Native        |
+-------------+------------------------------------+---------------+--------------+---------------+



I0000 00:00:1761084645.705613  124927 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3993 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1761084645.716883  124927 cuda_executor.cc:479] failed to allocate 3.90GiB (4186963968 bytes) from device: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
I0000 00:00:1761084645.761962  124927 gpu_device.cc:2019] Created device /device:GPU:0 with 3993 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [4]:

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models # type: ignore
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import Sequence # type: ignore
from tensorflow.keras.callbacks import ModelCheckpoint # type: ignore

# -------------------------
# Constants
# -------------------------

dataset_detail=np.load("/mnt/Extra/Project_Storage/stereo_ML_dataset/org/"+"detail.npy", allow_pickle=True)
dataset_detail=dataset_detail.tolist()

patch_shape = (dataset_detail[0],dataset_detail[1])

target = min(int(dataset_detail[2]*0.8),int(2**17+1))

Dmax = dataset_detail[3]

resize_fraction = dataset_detail[4]
resize_factor = 1/resize_fraction

BATCH_SIZE = 8
EPOCHS = 250


In [5]:

org_path = "/mnt/Extra/Project_Storage/stereo_ML_dataset/"
dataset_org_path = "/mnt/Extra/Project_Storage/stereo_ML_dataset/train/"

# -------------------------
# Load memmap datasets
# -------------------------
left_patch_memmap = dataset_org_path + "left_patch.dat"
right_strip_memmap = dataset_org_path + "right_strip.dat"
patch_disparity_memmap = dataset_org_path + "patch_disp.dat"

model_location = org_path+"model/"

left_patches  = np.memmap(left_patch_memmap, dtype=np.uint8, mode='r', shape=(target, patch_shape[0], patch_shape[1]))
right_strips  = np.memmap(right_strip_memmap, dtype=np.uint8, mode='r', shape=(target, Dmax, patch_shape[0], patch_shape[1] ))
median_disp   = np.memmap(patch_disparity_memmap, dtype=np.int16, mode='r', shape=(target,))





In [6]:
# -------------------------
# Batch Generator (already pre-shaped inputs)
# -------------------------
class BatchGenerator(Sequence):
    def __init__(self, left_mm, right_mm, disp_mm, indices,
                 batch_size, patch_shape, Dmax, **kwargs):
        super().__init__(**kwargs)
        self.left_mm = left_mm
        self.right_mm = right_mm
        self.disp_mm = disp_mm
        self.indices = indices
        self.batch_size = batch_size
        self.patch_shape = patch_shape
        self.Dmax = Dmax

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        # Take inputs directly
        Xl = self.left_mm[batch_idx].astype('float16') / 255.0
        Xl = np.expand_dims(Xl, axis=-1)  # (batch, n, m, 1)

        Xr = self.right_mm[batch_idx].astype('float16') / 255.0  # (batch, Dmax, n, m)
        Xr = np.expand_dims(Xr, axis=-1)  # (batch, Dmax, n, m, 1)

        y = self.disp_mm[batch_idx].astype('int16')  # already correct, no inversion

        return (Xl, Xr), y


# -------------------------
# Build Generators
# -------------------------
all_indices = np.arange(target)
idx_train, idx_val = train_test_split(all_indices, test_size=0.1, random_state=42)

train_gen = BatchGenerator(
    left_patches, right_strips, median_disp,
    idx_train, batch_size=BATCH_SIZE,
    patch_shape=patch_shape, Dmax=Dmax
)

val_gen = BatchGenerator(
    left_patches, right_strips, median_disp,
    idx_val, batch_size=BATCH_SIZE,
    patch_shape=patch_shape, Dmax=Dmax
)


In [7]:
# -------------------------
# Imports
# -------------------------
import tensorflow as tf
from tensorflow.keras import layers, models
import os


# -------------------------
# Checkpoint callback 
# -------------------------
class Checkpoint(tf.keras.callbacks.Callback):
    def __init__(self, model_location, save_every=4):
        super().__init__()
        self.model_location = model_location
        self.save_every = save_every
        os.makedirs(model_location, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.save_every == 0:
            filename = os.path.join(self.model_location, f"model_epoch_{epoch+1:02d}.keras")
            self.model.save(filename)
            print(f"\n Saved model checkpoint: {filename}\n")


# -------------------------
# GPU Memory Logger
# -------------------------
class GPUMemoryLogger(tf.keras.callbacks.Callback):
    def __init__(self, device_index=0):
        super().__init__()
        self.device_index = device_index

    def on_train_batch_end(self, batch, logs=None):
        try:
            mem_info = tf.config.experimental.get_memory_info(f'GPU:{self.device_index}')
            used = mem_info['current'] / (1024 ** 2)
            peak = mem_info['peak'] / (1024 ** 2)
            print(f"Batch {batch}: GPU{self.device_index} memory used = {used:.1f} MB | peak = {peak:.1f} MB")
        except Exception as e:
            print(f"Could not fetch GPU info: {e}")
            
import subprocess
import re
import tensorflow as tf

# -------------------------
# GPU Utilization Logger
# -------------------------
class GPUUtilizationLogger(tf.keras.callbacks.Callback):
    def __init__(self, device_index=0):
        super().__init__()
        self.device_index = device_index

    def on_train_batch_end(self, batch, logs=None):
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=utilization.gpu', '--format=csv,nounits,noheader', '-i', str(self.device_index)],
                stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
            )
            if result.returncode == 0:
                usage = int(result.stdout.strip())
                print(f"Batch {batch}: GPU{self.device_index} utilization = {usage}%")
            else:
                print(f"Batch {batch}: Could not fetch GPU utilization: {result.stderr.strip()}")
        except Exception as e:
            print(f"Could not fetch GPU utilization: {e}")


checkpoint_cb = Checkpoint(model_location, save_every=5)
gpu_logger = GPUMemoryLogger()
gpu_util = GPUUtilizationLogger()


In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts

# -------------------------
# Patch Encoder as Layer
# -------------------------
class PatchEncoder(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = layers.Conv2D(16, 3, padding='same', activation='relu')
        self.conv2 = layers.Conv2D(32, 3, padding='same', activation='relu')
        self.gap = layers.GlobalAveragePooling2D()

    def call(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.gap(x)
        return x  # (batch, feature_dim)

    def compute_output_shape(self, input_shape):
        batch_size = input_shape[0]
        return (batch_size, 32)

# -------------------------
# L2 Similarity Layer as Layer
# -------------------------
class L2Similarity(layers.Layer):
    def call(self, inputs):
        left_feat, right_feats = inputs  # left: (B, F), right: (B, D, F)
        # Broadcasting subtraction: left_feat[:, None, :] is shape (B, 1, F)
        diff = right_feats - left_feat[:, None, :]  # shape (B, D, F)
        sim = -tf.reduce_sum(tf.square(diff), axis=-1)  # (B, D)
        return sim


# -------------------------
# Build Disparity Selector Model
# -------------------------
def build_disparity_selector(patch_shape, max_disp):
    h, w = patch_shape
    left_input = layers.Input(shape=(h, w, 1), name="left_patch")
    right_input = layers.Input(shape=(max_disp, h, w, 1), name="right_patches")

    encoder = PatchEncoder()
    left_feat = encoder(left_input)
    right_feats = layers.TimeDistributed(encoder)(right_input)

    # similarity logits for all disparities
    sim_logits = L2Similarity()([left_feat, right_feats])  # (batch, max_disp)

    model = models.Model(inputs=[left_input, right_input], outputs=sim_logits, name="StereoDisparityClassifier")
    return model

# -------------------------
# Compile Model
# -------------------------
def compile_disparity_model(model):
    lr_schedule = CosineDecayRestarts(initial_learning_rate=1e-4, first_decay_steps=20000)
    optimizer = AdamW(learning_rate=lr_schedule, weight_decay=5e-6)

    # Use sparse categorical crossentropy since labels are integers (np.int16)
    
    def mae_disp(y_true, y_pred_logits):
        y_pred = tf.argmax(y_pred_logits, axis=-1, output_type=tf.int16)
        return tf.reduce_mean(tf.abs(tf.cast(y_true, tf.int16) - y_pred))

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy', mae_disp]
    )
    return model



# -------------------------
# Instantiate Model
# -------------------------
model = build_disparity_selector(patch_shape, Dmax)
model = compile_disparity_model(model)
model.summary()


Model: "StereoDisparityClassifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ left_patch          │ (None, 16, 16, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ right_patches       │ (None, 700, 16,   │          0 │ -                 │
│ (InputLayer)        │ 16, 1)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_encoder       │ (None, 32)        │      4,800 │ left_patch[0][0]  │
│ (PatchEncoder)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 700, 32)   │      4,800 │ right_patches[0]… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ l2_similarity       │ (None, 700)       │          0 │ patch_encoder[0]… │
│ (L2Similarity)      │                   │            │ time_distributed… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,800 (18.75 KB)

 Trainable params: 4,800 (18.75 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:

# -------------------------
# Callbacks
# -------------------------
early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

# -------------------------
# Training
# -------------------------
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    verbose=1,
    callbacks=[checkpoint_cb, early_stop_cb]
)

Epoch 1/250


I0000 00:00:1761084687.322399  125057 service.cc:152] XLA service 0x77a9f0006600 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761084687.322417  125057 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2025-10-22 03:41:30.042290: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1761084705.710518  125057 cuda_dnn.cc:529] Loaded cuDNN version 90701
I0000 00:00:1761084740.080840  125057 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


14746/14746 ━━━━━━━━━━━━━━━━━━━━ 938s 57ms/step - accuracy: 0.0322 - loss: 6.5521 - mae_disp: 99.1941 - val_accuracy: 0.0351 - val_loss: 6.5511 - val_mae_disp: 99.5516
Epoch 2/250
14746/14746 ━━━━━━━━━━━━━━━━━━━━ 727s 49ms/step - accuracy: 0.0322 - loss: 6.5521 - mae_disp: 99.1941 - val_accuracy: 0.0351 - val_loss: 6.5511 - val_mae_disp: 99.5516
Epoch 3/250
14746/14746 ━━━━━━━━━━━━━━━━━━━━ 733s 50ms/step - accuracy: 0.0322 - loss: 6.5521 - mae_disp: 99.1941 - val_accuracy: 0.0351 - val_loss: 6.5511 - val_mae_disp: 99.5516
Epoch 4/250
14746/14746 ━━━━━━━━━━━━━━━━━━━━ 734s 50ms/step - accuracy: 0.0322 - loss: 6.5521 - mae_disp: 99.1941 - val_accuracy: 0.0351 - val_loss: 6.5511 - val_mae_disp: 99.5516
Epoch 5/250
14746/14746 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.0325 - loss: 6.5511 - mae_disp: 97.9198
 Saved model checkpoint: /mnt/Extra/Project_Storage/stereo_ML_dataset/model/model_epoch_05.keras

14746/14746 ━━━━━━━━━━━━━━━━━━━━ 735s 50ms/step - accuracy: 0.0322 - loss: 6.5521 

In [10]:

import numpy as np

# Save
np.save(model_location+"history.npy", history.history)

model.save(model_location+"best_model.keras")
